<a href="https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI-Week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [12]:
from datasets import load_dataset
from huggingface_hub import login

login()

In [8]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

df = pd.DataFrame(list(ds.take(10000)))

numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0)

df["CTR"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

print("Rows:", len(df))

print(
    "\nBaseline rule:"
    "\nPrioritise pages that have meaningful search exposure, "
    "\nreasonable search visibility, and a relatively weak observed CTR."
)

print("\nReason codes:")
print("high_exposure")
print("visible_low_ctr")
print("position_opportunity")
print("monitor")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows: 10000

Baseline rule:
Prioritise pages that have meaningful search exposure, 
reasonable search visibility, and a relatively weak observed CTR.

Reason codes:
high_exposure
visible_low_ctr
position_opportunity
monitor


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [9]:
# Build transparent baseline components.

# 1. Exposure score.
# Log transform reduces the influence of very large impression counts.
exposure_log = np.log1p(df["gsc_impressions"])

if exposure_log.max() > exposure_log.min():
    df["exposure_score"] = (
        (exposure_log - exposure_log.min()) /
        (exposure_log.max() - exposure_log.min())
    )
else:
    df["exposure_score"] = 0.0


# 2. Position opportunity.
# Lower average position number = better visibility.
position = df["gsc_avg_position"].clip(lower=1, upper=100)

position_rank = position.rank(
    method="average",
    pct=True
)

df["position_score"] = (
    1 - position_rank
).fillna(0)

df.loc[
    df["gsc_avg_position"] <= 0,
    "position_score"
] = 0


# 3. CTR opportunity.
# Lower CTR + meaningful exposure = stronger review signal.
df["ctr_opportunity"] = (
    (1 - df["CTR"].clip(0, 1)) *
    df["exposure_score"]
)


# Transparent weighted rule.
df["baseline_action_score"] = (
    0.45 * df["exposure_score"] +
    0.30 * df["position_score"] +
    0.25 * df["ctr_opportunity"]
)

df["baseline_action_score"] = (
    df["baseline_action_score"]
    .clip(0, 1)
)


def reason_code(row):
    reasons = []

    if row["gsc_impressions"] >= 10:
        reasons.append("high_exposure")

    if (
        row["gsc_impressions"] >= 10
        and row["CTR"] < 0.02
    ):
        reasons.append("visible_low_ctr")

    if (
        row["gsc_avg_position"] > 0
        and row["gsc_avg_position"] <= 20
    ):
        reasons.append("position_opportunity")

    if not reasons:
        reasons.append("monitor")

    return "|".join(reasons)


df["reason_code"] = df.apply(
    reason_code,
    axis=1
)

df = df.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

df["baseline_rank"] = np.arange(
    1,
    len(df) + 1
)

print(
    df[
        [
            "baseline_rank",
            "baseline_action_score",
            "reason_code",
            "gsc_impressions",
            "gsc_avg_position",
            "CTR"
        ]
    ].head(20).to_string(index=False)
)

 baseline_rank  baseline_action_score                                        reason_code  gsc_impressions  gsc_avg_position      CTR
             1               0.975710 high_exposure|visible_low_ctr|position_opportunity              424          1.099057 0.000000
             2               0.940645 high_exposure|visible_low_ctr|position_opportunity              324          1.935185 0.000000
             3               0.932285                 high_exposure|position_opportunity              506          7.045455 0.021739
             4               0.930161 high_exposure|visible_low_ctr|position_opportunity              303          1.811881 0.009901
             5               0.930023 high_exposure|visible_low_ctr|position_opportunity              305          1.586885 0.016393
             6               0.920045 high_exposure|visible_low_ctr|position_opportunity              466          7.500000 0.004292
             7               0.910980                 high_exposure|p

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [10]:
# Top-20 review table.

top20 = df.head(20).copy()


def confidence_note(row):
    if row["gsc_impressions"] >= 100:
        return "Higher exposure; observed CTR is more informative."
    elif row["gsc_impressions"] >= 10:
        return "Moderate exposure; review before acting."
    else:
        return "Low exposure; treat the score as tentative."


def wrong_if(row):
    if row["gsc_impressions"] < 10:
        return "Low exposure may make CTR unstable."
    if row["gsc_avg_position"] <= 0:
        return "Missing/invalid position reduces confidence."
    return "The observed signal may not identify the underlying cause."


top20["action"] = "Review content/search context"
top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)
top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)

display(
    top20[
        [
            "baseline_rank",
            "content_hash_id",
            "baseline_action_score",
            "reason_code",
            "action",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,baseline_rank,content_hash_id,baseline_action_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_213eb91f21a43550,0.975710,high_exposure|visible_low_ctr|position_opportu...,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
1,2,content_213eb91f21a43550,0.940645,high_exposure|visible_low_ctr|position_opportu...,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
2,3,content_4e8d1e11f60fe6ba,0.932285,high_exposure|position_opportunity,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
3,4,content_f94fe855380e150f,0.930161,high_exposure|visible_low_ctr|position_opportu...,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
4,5,content_f94fe855380e150f,0.930023,high_exposure|visible_low_ctr|position_opportu...,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
5,6,content_4e8d1e11f60fe6ba,0.920045,high_exposure|visible_low_ctr|position_opportu...,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
6,7,content_f94fe855380e150f,0.910980,high_exposure|position_opportunity,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
7,8,content_f94fe855380e150f,0.901823,high_exposure|position_opportunity,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
8,9,content_f94fe855380e150f,0.901438,high_exposure|position_opportunity,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...
9,10,content_f94fe855380e150f,0.898562,high_exposure|position_opportunity,Review content/search context,Higher exposure; observed CTR is more informat...,The observed signal may not identify the under...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [11]:
# Weak-pick analysis and leakage check.

print("WEAK PICK CHECK")
print("=" * 70)

high_score_cutoff = df[
    "baseline_action_score"
].quantile(0.90)

weak_picks = df[
    (df["baseline_action_score"] >= high_score_cutoff) &
    (df["gsc_impressions"] < 10)
]

print(
    "High-score rows with fewer than 10 impressions:",
    len(weak_picks)
)

display(
    weak_picks[
        [
            "baseline_rank",
            "baseline_action_score",
            "gsc_impressions",
            "gsc_avg_position",
            "CTR",
            "reason_code"
        ]
    ].head(10)
)

print("\nLEAKAGE CHECK")
print("=" * 70)

# These are intentionally not used to calculate the score.
for col in [
    "gsc_clicks",
    "trend_direction",
    "trend_pct"
]:
    print(f"{col}: excluded from scoring features")

print("\nScoring inputs:")
print([
    "gsc_impressions",
    "gsc_avg_position",
    "observed CTR"
])


# Write required CSV.
output_dir = Path("work/outputs")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir /
    "baseline_action_score.csv"
)

df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "baseline_rank",
        "baseline_action_score",
        "exposure_score",
        "position_score",
        "ctr_opportunity",
        "reason_code",
        "gsc_impressions",
        "gsc_avg_position",
        "CTR"
    ]
].to_csv(
    output_path,
    index=False
)

print(
    f"\nSaved required output: {output_path}"
)

WEAK PICK CHECK
High-score rows with fewer than 10 impressions: 0


,baseline_rank,baseline_action_score,gsc_impressions,gsc_avg_position,CTR,reason_code



LEAKAGE CHECK
gsc_clicks: excluded from scoring features
trend_direction: excluded from scoring features
trend_pct: excluded from scoring features

Scoring inputs:
['gsc_impressions', 'gsc_avg_position', 'observed CTR']

Saved required output: work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.